In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn file
file_path = '/content/drive/MyDrive/Thesis/df_final_relations_depen1.csv'

# 3. Đọc file CSV
try:
    # Sử dụng encoding='utf-8-sig' để đảm bảo không lỗi font tiếng Việt
    df_final_relations_depen1 = pd.read_csv(file_path, encoding='utf-8-sig')

    print("✅ Đã đọc file thành công!")
    print(f"Số lượng dòng: {df_final_relations_depen1.shape[0]}")
    print(f"Số lượng cột: {df_final_relations_depen1.shape[1]}")

    # Hiển thị 5 dòng đầu để kiểm tra
    display(df_final_relations_depen1.head())

except FileNotFoundError:
    print(f"❌ Không tìm thấy file tại đường dẫn: {file_path}")
    print("Bạn hãy kiểm tra lại tên thư mục 'Thesis' hoặc tên file có đúng chính tả chưa nhé.")
except Exception as e:
    print(f"⚠️ Có lỗi xảy ra: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã đọc file thành công!
Số lượng dòng: 27030
Số lượng cột: 6


,Hotel_Name,Relation,Object,Object_Type,Gap_Words,Context
0,Havana Nha Trang Hotel,SUITABLE_FOR,families,TARGET,7,5 5 Highly Recommended Perfect Stay for Famili...
1,Havana Nha Trang Hotel,SUITABLE_FOR,travellers,TARGET,10,5 5 Highly Recommended Perfect Stay for Famili...
2,Havana Nha Trang Hotel,LOCATED_IN,beach,LOC,72,5 5 Highly Recommended Perfect Stay for Famili...
3,Havana Nha Trang Hotel,NEARBY,beach,LOC,85,5 5 Highly Recommended Perfect Stay for Famili...
4,Havana Nha Trang Hotel,HAS_FACILITY,massage,FACILITY,96,5 5 Highly Recommended Perfect Stay for Famili...


In [ ]:
LAimport json
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq
from google.colab import drive
from IPython.display import display

# ==============================================================================
# 1. KẾT NỐI & CẤU HÌNH TEST 100 DÒNG
# ==============================================================================
drive.mount('/content/drive')
client = Groq(api_key="")

SELECTED_MODEL = 'openai/gpt-oss-120b'
BATCH_SIZE = 4
OUTPUT_TEST_FILE = "/content/drive/MyDrive/df_relation_TEST_100_RESULTS.csv"

# ==============================================================================
# 2. HÀM XỬ LÝ (GIỮ NGUYÊN PROMPT CỦA ÔNG)
# ==============================================================================
def process_relations_batch_groq(batch_data):
    input_json_str = json.dumps(batch_data, ensure_ascii=False)

    prompt = f"""
    You are an expert NLP Data Annotator. Your task is to validate and CORRECT hotel relationship labels.

1. HAS_FACILITY: Amenities provided WITHIN the hotel (pool, gym, wifi, restaurant, food, etc.).
   - STRICT RULE: DO NOT label if the context is negative (e.g., "no pool", "not have wifi", "without gym"). These must be marked as INVALID.
2. LOCATED_IN: The administrative location where the hotel is physically situated (Street, Ward, District, City, Province).
   - STRICT RULE: ONLY include geographic/administrative entities. DO NOT where it's located và do not include nearby attractions, malls, or landmarks here.
3. NEARBY: Landmarks, attractions, or locations CLOSE to the hotel but NOT its administrative address.
   - STRICT RULE: DO NOT include city, district, or province names in this category (those belong to LOCATED_IN).
4. HAS_PRICE_INFO: Costs, rates, or specific money details.
   - ADDITION: Include adjectives describing price levels (e.g., "cheap", "expensive", "affordable", "budget-friendly").
5. SUITABLE_FOR: Target audience or purpose of stay (families, couples, solo travelers, business trips).
    TASK:
    1. Check if the current 'Relation' is correct based on the 'Context'.
    2. If correct, set status='VALID' and keep the 'Relation'.
    3. If incorrect but the context supports a DIFFERENT valid relation, set status='CORRECTED' and update 'Relation'.
    4. If it's completely wrong or negated (e.g., "no pool"), set status='INVALID'.

    DATA:
    {input_json_str}

    Return EXACTLY a JSON object with a "results" key containing objects with: "id", "status", "final_relation".
    Do not output any markdown formatting, only the JSON.
    """

    for attempt in range(3):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=SELECTED_MODEL,
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            result_json = json.loads(chat_completion.choices[0].message.content)
            res_map = {int(item["id"]): (item["status"], item["final_relation"]) for item in result_json["results"]}

            stats, rels = [], []
            for item in batch_data:
                s, r = res_map.get(item["id"], ("ERROR", item["Relation"]))
                stats.append(s); rels.append(r)
            return stats, rels, "SUCCESS"
        except Exception as e:
            time.sleep(15 if "429" in str(e) else 5)
            if attempt == 2: return ["ERROR"]*len(batch_data), [r["Relation"] for r in batch_data], "FAIL"

# ==============================================================================
# 3. LẤY MẪU 100 DÒNG & CHẠY
# ==============================================================================
# Giả sử df_final_relations_depen1 là dataframe gốc của ông
df_test_100 = df_final_relations_depen1.sample(n=100, random_state=42).copy()
df_test_100['Original_Relation'] = df_test_100['Relation']
df_test_100['LLM_Status'] = "PENDING"
df_test_100 = df_test_100.reset_index(drop=True)

final_statuses = []
final_relations = []

print(f"🚀 Đang chạy thử 100 dòng với model {SELECTED_MODEL}...")

for i in tqdm(range(0, len(df_test_100), BATCH_SIZE)):
    batch_df = df_test_100.iloc[i:i+BATCH_SIZE]
    batch_payload = [{"id": idx, "Context": row["Context"], "Hotel_Name": row["Hotel_Name"],
                      "Relation": row["Relation"], "Object": row["Object"]}
                     for idx, row in batch_df.iterrows()]

    batch_statuses, batch_relations, _ = process_relations_batch_groq(batch_payload)
    final_statuses.extend(batch_statuses)
    final_relations.extend(batch_relations)

    time.sleep(3) # Nghỉ để tránh Rate Limit

# Gán kết quả và hiển thị
df_test_100['LLM_Status'] = final_statuses
df_test_100['Relation'] = final_relations

print("\n--- 📊 KẾT QUẢ TEST 100 DÒNG ---")
display(df_test_100[['Hotel_Name', 'Original_Relation', 'Relation', 'Object', 'LLM_Status', 'Context']].head(20))

# Lưu file test để ông soi cho kỹ
df_test_100.to_csv(OUTPUT_TEST_FILE, index=False, encoding='utf-8-sig')
print(f"✅ Đã lưu kết quả test tại: {OUTPUT_TEST_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Đang chạy thử 100 dòng với model openai/gpt-oss-120b...


100%|██████████| 25/25 [19:51<00:00, 47.68s/it]


--- 📊 KẾT QUẢ TEST 100 DÒNG ---


,Hotel_Name,Original_Relation,Relation,Object,LLM_Status,Context
0,Beach Hotel,HAS_FACILITY,HAS_FACILITY,seafront,ERROR,Im glad that I booked Platinum Beach Hotel Dan...
1,Beach Hotel,LOCATED_IN,LOCATED_IN,Da Nang,ERROR,I had an absolutely fantastic stay at SANA Dan...
2,LA VELA Saigon Hotel,HAS_FACILITY,HAS_FACILITY,attractions,ERROR,I had a fantastic stay at LA VELA Saigon Hotel...
3,West,HAS_FACILITY,HAS_FACILITY,room,ERROR,Wild Lotus Hotel Xuan Dieu Review Great Locati...
4,Beachfront Hotel,HAS_FACILITY,HAS_FACILITY,room,ERROR,Wed heard that this hotel was among the most b...
5,Ha Long,LOCATED_IN,LOCATED_IN,Old Quarter,ERROR,A hotel in Old Quarter is a great jumping off ...
6,Clay,HAS_PRICE_INFO,HAS_PRICE_INFO,Price,ERROR,This is my first time checking in at The Clay ...
7,Phuc Long,HAS_FACILITY,HAS_FACILITY,view,ERROR,I just had a wonderful vacation at Phuc Long L...
8,Bich Duyen,HAS_FACILITY,HAS_FACILITY,elevator,ERROR,We very much enjoyed our stay at the Bich Duye...
9,La Renta Hotel,NEARBY,NEARBY,Lake,ERROR,Overview La Renta Hotel is a 3 star hotel loca...


✅ Đã lưu kết quả test tại: /content/drive/MyDrive/Thesis/df_relation_TEST_100_RESULTS.csv


In [ ]:
pip install pandas openpyxl groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.0 MB/s eta 0:00:00


In [ ]:
import json
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq
from google.colab import drive
from IPython.display import display

# ==============================================================================
# 1. KẾT NỐI & CẤU HÌNH TEST 100 DÒNG
# ==============================================================================
drive.mount('/content/drive')
client = Groq(api_key="")

SELECTED_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'
BATCH_SIZE = 4
OUTPUT_TEST_FILE = "/content/drive/MyDrive/Thesis/df_relation_TEST_100_RESULTS.csv"

# ==============================================================================
# 2. HÀM XỬ LÝ (GIỮ NGUYÊN PROMPT CỦA ÔNG)
# ==============================================================================
def process_relations_batch_groq(batch_data):
    input_json_str = json.dumps(batch_data, ensure_ascii=False)

    prompt = f"""
    You are an expert NLP Data Annotator. Your task is to validate and CORRECT hotel relationship labels.

1. HAS_FACILITY: Amenities provided WITHIN the hotel (pool, gym, wifi, restaurant, food, etc.).
   - STRICT RULE: DO NOT label if the context is negative (e.g., "no pool", "not have wifi", "without gym"). These must be marked as INVALID.
2. LOCATED_IN: The administrative location where the hotel is physically situated (Street, Ward, District, City, Province).
   - STRICT RULE: ONLY include geographic/administrative entities. DO NOT where it's located và do not include nearby attractions, malls, or landmarks here.
3. NEARBY: Landmarks, attractions, or locations CLOSE to the hotel but NOT its administrative address.
   - STRICT RULE: DO NOT include city, district, or province names in this category (those belong to LOCATED_IN).
4. HAS_PRICE_INFO: Costs, rates, or specific money details.
   - ADDITION: Include adjectives describing price levels (e.g., "cheap", "expensive", "affordable", "budget-friendly").
5. SUITABLE_FOR: Target audience or purpose of stay (families, couples, solo travelers, business trips).
    TASK:
    1. Check if the current 'Relation' is correct based on the 'Context'.
    2. If correct, set status='VALID' and keep the 'Relation'.
    3. If incorrect but the context supports a DIFFERENT valid relation, set status='CORRECTED' and update 'Relation'.
    4. If it's completely wrong or negated (e.g., "no pool"), set status='INVALID'.

    DATA:
    {input_json_str}

    Return EXACTLY a JSON object with a "results" key containing objects with: "id", "status", "final_relation".
    Do not output any markdown formatting, only the JSON.
    """

    for attempt in range(3):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=SELECTED_MODEL,
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            result_json = json.loads(chat_completion.choices[0].message.content)
            res_map = {int(item["id"]): (item["status"], item["final_relation"]) for item in result_json["results"]}

            stats, rels = [], []
            for item in batch_data:
                s, r = res_map.get(item["id"], ("ERROR", item["Relation"]))
                stats.append(s); rels.append(r)
            return stats, rels, "SUCCESS"
        except Exception as e:
            time.sleep(15 if "429" in str(e) else 5)
            if attempt == 2: return ["ERROR"]*len(batch_data), [r["Relation"] for r in batch_data], "FAIL"

# ==============================================================================
# 3. LẤY MẪU 100 DÒNG & CHẠY
# ==============================================================================
# Giả sử df_final_relations_depen1 là dataframe gốc của ông
df_test_100 = df_final_relations_depen1.sample(n=100, random_state=42).copy()
df_test_100['Original_Relation'] = df_test_100['Relation']
df_test_100['LLM_Status'] = "PENDING"
df_test_100 = df_test_100.reset_index(drop=True)

final_statuses = []
final_relations = []

print(f"🚀 Đang chạy thử 100 dòng với model {SELECTED_MODEL}...")

for i in tqdm(range(0, len(df_test_100), BATCH_SIZE)):
    batch_df = df_test_100.iloc[i:i+BATCH_SIZE]
    batch_payload = [{"id": idx, "Context": row["Context"], "Hotel_Name": row["Hotel_Name"],
                      "Relation": row["Relation"], "Object": row["Object"]}
                     for idx, row in batch_df.iterrows()]

    batch_statuses, batch_relations, _ = process_relations_batch_groq(batch_payload)
    final_statuses.extend(batch_statuses)
    final_relations.extend(batch_relations)

    time.sleep(3) # Nghỉ để tránh Rate Limit

# Gán kết quả và hiển thị
df_test_100['LLM_Status'] = final_statuses
df_test_100['Relation'] = final_relations

print("\n--- 📊 KẾT QUẢ TEST 100 DÒNG ---")
display(df_test_100[['Hotel_Name', 'Original_Relation', 'Relation', 'LLM_Status', 'Context']].head(20))

# Lưu file test để ông soi cho kỹ
df_test_100.to_csv(OUTPUT_TEST_FILE, index=False, encoding='utf-8-sig')
print(f"✅ Đã lưu kết quả test tại: {OUTPUT_TEST_FILE}")

In [ ]:
import json
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq
from google.colab import drive
from IPython.display import display

# ==============================================================================
# 1. KẾT NỐI & CẤU HÌNH TEST 100 DÒNG
# ==============================================================================
drive.mount('/content/drive')
client = Groq(api_key="")

SELECTED_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'


# Tên file Đầu vào / Đầu ra (Sửa lại cho khớp với tên file trên máy ông)
input_file = "/content/drive/MyDrive/Thesis/Sentiment_Manual_Labeling.xlsx"
output_file = "/content/drive/MyDrive/Thesis/Sentiment_Auto_Labeled_Output.xlsx"

# ==========================================
# PROMPT BỌC THÉP
# ==========================================
SYSTEM_PROMPT = """You are an expert Aspect-Based Sentiment Analysis (ABSA) system.
Task: Determine the sentiment of the specific `Object` based on the review `Context`.
Constraint: You MUST output STRICTLY ONLY ONE WORD: POSITIVE, NEGATIVE, or NEUTRAL. Do not output any periods, punctuation, or explanations. If you are unsure, output NEUTRAL.

Examples:
Context: "The Lumiere has a beautiful reception area." | Object: "reception" -> Answer: POSITIVE
Context: "The charge for breakfast was a rip off." | Object: "charge" -> Answer: NEGATIVE
Context: "Our stay at Bella MERF was suitable for our family." | Object: "family" -> Answer: POSITIVE"""

# ==========================================
# THỰC THI CHÍNH
# ==========================================
print("📂 Đang đọc file Excel...")
try:
    df = pd.read_excel(input_file)
except FileNotFoundError:
    print(f"❌ Không tìm thấy file {input_file}. Ông kiểm tra lại tên file nhé!")
    exit()

if 'Sentiment' not in df.columns:
    df['Sentiment'] = ""
df['Sentiment'] = df['Sentiment'].astype(str)

total_rows = len(df)
print(f"🚀 Bắt đầu dán nhãn tự động cho {total_rows} dòng...\n")

for index, row in df.iterrows():
    # Bỏ qua những dòng đã được dán nhãn tay từ trước
    current_sentiment = str(row['Sentiment']).strip().upper()
    if current_sentiment in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
        continue

    context = row['Context']
    obj = row['Object']

    user_prompt = f"Context: '{context}' | Object: '{obj}' -> Answer:"

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model='meta-llama/llama-4-scout-17b-16e-instruct', # Dùng con Llama 3 8B vì nó chạy nhanh như điện
                #'meta-llama/llama-4-scout-17b-16e-instruct'
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0, # Ép nó trả lời máy móc nhất có thể
                max_tokens=10
            )

            # Lấy kết quả và dọn dẹp
            answer = response.choices[0].message.content.strip().upper()

            # Quét dọn lần cuối: Ép về đúng 3 từ khóa
            if "POSITIVE" in answer: final_label = "POSITIVE"
            elif "NEGATIVE" in answer: final_label = "NEGATIVE"
            else: final_label = "NEUTRAL"

            # Ghi vào Dataframe
            df.at[index, 'Sentiment'] = final_label
            print(f"✅ [{index + 1}/{total_rows}] Object: '{obj}' -> {final_label}")

            # Ngủ 2 giây để lách luật Rate Limit của Groq (Tầm 20 phút là xong 600 dòng)
            time.sleep(2)
            break # Thành công thì thoát vòng lặp Retry để đi tiếp dòng sau

        except Exception as e:
            print(f"⚠️ Bị vấp ở dòng {index + 1} (Đang thử lại {attempt+1}/{max_retries}): {e}")
            time.sleep(5) # Nghỉ giải lao 5s rồi chạy lại dòng này

# ==========================================
# LƯU FILE KẾT QUẢ
# ==========================================
df.to_excel(output_file, index=False)
print("\n" + "="*50)
print(f"🎉 ĐÃ HOÀN THÀNH TẤT CẢ! File kết quả lưu tại: {output_file}")
print("="*50)

Mounted at /content/drive
📂 Đang đọc file Excel...
🚀 Bắt đầu dán nhãn tự động cho 600 dòng...

✅ [1/600] Object: 'reception' -> POSITIVE
✅ [2/600] Object: 'family' -> POSITIVE
✅ [3/600] Object: 'family' -> POSITIVE
✅ [4/600] Object: 'reception' -> POSITIVE
✅ [5/600] Object: 'housekeeping' -> POSITIVE
✅ [6/600] Object: 'charge' -> NEGATIVE
✅ [7/600] Object: 'amenities' -> POSITIVE
✅ [8/600] Object: 'families' -> POSITIVE
✅ [9/600] Object: 'Breakfast' -> POSITIVE
✅ [10/600] Object: 'charge' -> POSITIVE
✅ [11/600] Object: 'housekeeping' -> POSITIVE
✅ [12/600] Object: 'shower' -> POSITIVE
✅ [13/600] Object: 'elevator' -> POSITIVE
✅ [14/600] Object: 'family' -> POSITIVE
✅ [15/600] Object: 'complimentary' -> NEGATIVE
✅ [16/600] Object: 'value' -> POSITIVE
✅ [17/600] Object: 'Pool' -> POSITIVE
✅ [18/600] Object: 'money' -> NEGATIVE
✅ [19/600] Object: 'sink' -> NEGATIVE
✅ [20/600] Object: 'shower' -> POSITIVE
✅ [21/600] Object: 'dining' -> POSITIVE
✅ [22/600] Object: 'travellers' -> POSITIVE
✅